# ⚽ Pipeline completo — Football Tracking → Eventos

**Cómo correrlo:** Activá GPU (Entorno de ejecución → GPU) y hacé **Ejecutar todo**.
La celda 1 instala todo y te pide **reiniciar una vez**: reiniciás y volvés a **Ejecutar todo**.
De ahí en más va solo (solo seleccionás el video cuando lo pida).

> Los *warnings* de pip sobre `pointpats/esda/spopt/...` son normales — esos paquetes no se usan.

---

### Qué se está validando en esta corrida

El clip de 3 min con la **acumulación de keypoints** (`KeypointBuffer` + `median_image_shift`).
Los tres números que deciden si cerró, en este orden:

| celda | número | hoy | objetivo |
|---|---|---|---|
| **5a** | span de cancha que cubren los keypoints | ~20 m | **60-100 m** |
| **5c** | bloque CALIBRACION | `MAL CALIBRADA` | `plausible` |
| **5c** | pelota dentro del área de penal | 58,6% | **~19,7%** |

Si 5a se queda en ~20 m, **no sigas**: el problema está en `median_image_shift()` y nada
río abajo cambia.


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una en Entorno de ejecución')


## 1. Instalar (una vez) + reiniciar

Instala dependencias, actualiza ultralytics (para el modelo entrenado), deja Pillow limpio y fija
`numpy<2.1` (lo necesita el clasificador de equipos). Al terminar te pide reiniciar.


In [ ]:
import os
REPO_DIR = '/content/ncf_event_tracker'
FLAG = '/content/.setup_done'
BRANCH = 'events-model'   # rama con --frame-stride y el sidecar .meta.json

if not os.path.exists(FLAG):
    if not os.path.exists(REPO_DIR):
        !git clone -q --branch {BRANCH} https://github.com/pipachiesa/ncf_event_tracker.git {REPO_DIR}
    !pip install -q -r {REPO_DIR}/requirements.txt
    !pip install -q filterpy scipy
    !pip install -q -U ultralytics
    !pip install -q --force-reinstall --no-cache-dir pillow
    !pip install -q 'numpy<2.1'   # para el clasificador de equipos (numba)
    open(FLAG, 'w').close()
    print('\n' + '='*66)
    print('✅ INSTALADO. Ahora: Entorno de ejecución → REINICIAR entorno,')
    print('   y volvé a Ejecutar todo (esta celda se saltea sola).')
    print('='*66)
    raise SystemExit('Reiniciá el entorno y volvé a Ejecutar todo.')

%cd {REPO_DIR}
!git checkout -q {BRANCH} && git pull -q origin {BRANCH}   # asegura la rama correcta
import numpy, ultralytics
print('rama:', open(REPO_DIR+'/.git/HEAD').read().strip().split('/')[-1], '| numpy:', numpy.__version__, '| ultralytics:', ultralytics.__version__)
try:
    import numba; from sports.common.team import TeamClassifier
    print('✅ Entorno OK — clasificador de equipos disponible.')
except Exception as e:
    print(f'⚠️  Clasificador de equipos NO disponible ({type(e).__name__}). El pipeline correrá SIN equipos.')
assert os.path.exists('data_cleanup/main.py')

# El notebook SE DESINCRONIZA del repo: verificar que el pull trajo el codigo
# nuevo, no confiar en que "deberia estar".
!git --no-pager log --oneline -1
_main = open('data_cleanup/main.py').read()
for _n in ('class KeypointBuffer', 'def median_image_shift',
           'MIN_KEYPOINT_PITCH_X_CM'):
    assert _n in _main, f'❌ el pull NO trajo {_n} — el fix de keypoints no esta'
print('✅ acumulacion de keypoints presente en main.py')


## 2. Montar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR    = '/content/drive/MyDrive/football_analytics'
TRACKING_DIR = os.path.join(DRIVE_DIR, 'tracking_output')
EVENTS_DIR   = os.path.join(DRIVE_DIR, 'event_output')
for d in (DRIVE_DIR, TRACKING_DIR, EVENTS_DIR): os.makedirs(d, exist_ok=True)
print('Resultados en:', DRIVE_DIR)


## 3. Detector

Busca tu modelo entrenado en Drive y verifica que carga. Si no hay, usa el community `football`.


In [ ]:
import glob
from ultralytics import YOLO
hits = glob.glob(os.path.join(DRIVE_DIR, 'models', '**', 'best.pt'), recursive=True)
if hits:
    DETECTOR = hits[0]
    print('Modelo entrenado:', DETECTOR)
    print('  carga OK, clases:', YOLO(DETECTOR).names)
else:
    DETECTOR = 'football'
    print('⚠️ No hay modelo entrenado en Drive — usando community \'football\'')


## 4. Elegí el video (desde Google Drive)

Lo más simple: subí el `.mp4` a **MyDrive/football_analytics/videos/** y poné
solo el **nombre del archivo** en `VIDEO_FILE`. También podés pegar una ruta
completa de Drive. (Si dejás `VIDEO_FILE=''` cae al viejo modo "subir del navegador".)

In [ ]:
# Subí tu video a MyDrive/football_analytics/videos/ y poné su nombre acá.
# (o pegá una ruta completa de Drive; dejalo '' para subir del navegador)
VIDEO_FILE = 'spain-france-test3min.mp4'   # el clip de 3 min de validacion

VIDEOS_DIR = os.path.join(DRIVE_DIR, 'videos')
os.makedirs(VIDEOS_DIR, exist_ok=True)

if not VIDEO_FILE:
    from google.colab import files
    print('Seleccioná un .mp4 desde tu compu...')
    up = files.upload()
    VIDEO_PATH = os.path.abspath(list(up.keys())[0])
else:
    # ruta completa tal cual, o solo-nombre -> buscar en la carpeta de videos
    VIDEO_PATH = VIDEO_FILE if os.path.sep in VIDEO_FILE else os.path.join(VIDEOS_DIR, VIDEO_FILE)
    if not os.path.exists(VIDEO_PATH):
        hay = sorted(glob.glob(os.path.join(VIDEOS_DIR, '*.mp4')))
        listado = '\n'.join('  - ' + os.path.basename(v) for v in hay) or '  (carpeta vacía)'
        raise FileNotFoundError(
            f'No encuentro el video: {VIDEO_PATH}\n'
            f'Subí el .mp4 a {VIDEOS_DIR} y poné su nombre en VIDEO_FILE.\n'
            f'Videos que hay ahí ahora:\n{listado}')

VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print('Video:', VIDEO_PATH)

## 5. Tracking

Borra cualquier CSV viejo primero, así si el tracking falla lo ves (no arrastra datos viejos).

> **Dos modelos distintos a propósito:** el detector entrenado para **jugadores**, y el
> community `football` para el **balón** (el entrenado detecta el balón peor: 54% vs 78%).


In [ ]:
# BALL_CROP=True para que la corrida sea comparable con el baseline
# (`..._crop (3).csv` del 18-ago: pelota en area 58,6%). No cambiarlo en esta
# validacion, o no hay contra que comparar.
BALL_CROP = True
SUFIJO = '_crop' if BALL_CROP else '_base'

TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + SUFIJO + '.csv')
if os.path.exists(TRACKING_CSV): os.remove(TRACKING_CSV)

cmd = (
    f'python data_cleanup/main.py '
    f'--video "{VIDEO_PATH}" '
    f'--output "{TRACKING_DIR}" '
    f'--player-model "{DETECTOR}" '
    f'--ball-model football '   # el modelo entrenado detecta PEOR el balon (54% vs 78%)
    f'--imgsz 1280 '
    f'--pitch-imgsz 1280 '
    f'--ball-conf 0.1 '
    f'--ball-interp-gap 15 '
    f'--track-buffer 150 '
    f'--min-track-frames 12 '
    f'--pitch-model football-field '
    f'--homography-every 5 '
    f'--frame-stride 2 '
    + (f'--ball-crop' if BALL_CROP else '')
)
print(cmd, '\n')
# El log va a archivo ademas de a pantalla: la linea "Keypoints acumulados"
# queda sepultada entre las barras de tqdm y es EL numero de esta corrida.
LOG = '/content/track_log.txt'
!{cmd} 2>&1 | tee {LOG}
assert os.path.exists(TRACKING_CSV), '❌ El tracking FALLÓ (no se generó el CSV). Mirá el error de arriba.'
print('✅ Tracking CSV:', TRACKING_CSV)


## 5b. CHEQUEO CLAVE — ¿la pelota se teletransporta?

Test de aceptación de la corrida. Antes, el 24,5% de los movimientos de la
pelota eran **físicamente imposibles** (p90 = 417 m/s = 1502 km/h): el detector
elegía por confianza y saltaba al punto de penal. Con la selección por
continuidad (`_pick_ball`) esto tiene que desplomarse.

> **< 5% = arreglado.** Si sigue arriba del 15%, el fix no entró (¿hiciste
> `git pull` de la rama con el cambio?) y no tiene sentido etiquetar todavía.

In [ ]:
import csv, json

FPS = json.load(open(TRACKING_CSV.rsplit('.',1)[0] + '.meta.json'))['effective_fps']
# La cancha de SoccerPitchConfiguration es de 120 x 70 m, NO 105 x 68: reescalar
# a 105x68 achicaba todas las velocidades un 12,5%, asi que el umbral de 35 m/s
# dejaba pasar movimientos imposibles.
L_M, W_M = 120.0, 70.0

pts = []
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] != 'ball':
        continue
    x, y = float(r['X_Pitch']), float(r['Y_Pitch'])
    if x == 0 and y == 0:
        continue
    pts.append((int(r['Frame']), x/12000*L_M, y/7000*W_M))
pts.sort()

sp = []
for i in range(1, len(pts)):
    (f0,x0,y0), (f1,x1,y1) = pts[i-1], pts[i]
    if not 1 <= f1-f0 <= 3:
        continue
    sp.append((((x1-x0)**2 + (y1-y0)**2)**0.5) / ((f1-f0)/FPS))
sp.sort()

pct = lambda q: sp[int(q*(len(sp)-1))]
bad = 100*sum(1 for s in sp if s > 35)/len(sp)   # 35 m/s = tiro potente
print(f'detecciones de pelota: {len(pts)}')
print(f'velocidad p50 {pct(.5):5.1f} m/s | p90 {pct(.9):7.1f} | p99 {pct(.99):7.1f}')
print(f'\nMOVIMIENTOS IMPOSIBLES (>35 m/s): {bad:.1f}%   (antes del fix: 24.5%)')
print('✅ trayectoria sana' if bad < 5 else
      ('🟡 mejoró pero no del todo' if bad < 15 else
       '❌ el fix NO entró — revisá que el git pull traiga _pick_ball'))
cand = TRACKING_CSV.rsplit('.', 1)[0] + '_ball_candidates.csv'
if os.path.exists(cand):
    n = sum(1 for _ in open(cand)) - 1
    print(f'\ncandidatos de pelota guardados: {n} (para ball_viterbi.py) ✅')
else:
    print('\n❌ no se generaron candidatos — el git pull no trajo la version nueva')

## 5a. EL NÚMERO DE ESTA CORRIDA — ¿cuánta cancha cubren los keypoints?

`main.py` ahora **acumula keypoints entre refrescos**, arrastrándolos con el paneo de la
cámara, porque en 3 de cada 4 frames los únicos keypoints confiables son los del área de
penal izquierda: un parche de **20 m sobre una cancha de 120**. Ajustar la homografía a ese
parche da error de reproyección bajo *ahí* y cientos de metros de error en el resto.

> **60-100 m = funcionó. ~20 m = la compensación del paneo no agarró**, y en ese caso el
> problema está en `median_image_shift`, no en el resto del pipeline: no sigas con la
> corrida larga.


In [ ]:
# La linea la imprime main.py al terminar; la rescatamos del log porque queda
# sepultada entre las barras de tqdm.
import re

log = open(LOG, errors='ignore').read()
m = re.search(r'Keypoints acumulados al final:.*', log)
if not m:
    print('❌ no aparece la linea de keypoints acumulados en el log.')
    print('   O el pull no trajo el codigo nuevo, o el modelo de cancha no corrio.')
else:
    linea = m.group(0)
    print(linea, '\n')
    # "...cubriendo 85 x 60 m de cancha (un frame solo cubre ~20 x 55 m)":
    # el primer par es el acumulado, el segundo es el texto de referencia.
    span = re.search(r'cubriendo\s+(\d+)\s*x\s*(\d+)\s*m', linea)
    if not span:
        print('(no pude parsear el span; leelo a ojo en la linea de arriba)')
    else:
        span_x, span_y = int(span.group(1)), int(span.group(2))
        print(f'span en cancha: {span_x} x {span_y} m   (cancha 120 x 70)')
        if span_x >= 60:
            print('✅ FUNCIONO — los keypoints cubren cancha suficiente.')
        elif span_x >= 35:
            print('🟡 A MEDIAS — mejoro respecto de los 20 m pero sigue siendo poca')
            print('   cancha. Decide el bloque CALIBRACION de la celda 5c.')
        else:
            print('❌ SIGUE EN ~20 m — la compensacion del paneo NO agarro.')
            print('   El problema esta en median_image_shift(); no sigas con el')
            print('   partido completo, no cambia nada rio abajo.')

# Cuantas homografias se aceptaron: si el rechazo es ~96% el mapa esta congelado
# (paso, y fue un error caro: congelado NO puede saltar, asi que sacaba nota
# perfecta en estabilidad mientras ponia al arquero a 39 m de su arco).
h = re.search(r'Homografia:.*', log)
if h:
    print('\n' + h.group(0))
    pct = re.search(r'\(([\d.]+)% rechazo\)', h.group(0))
    if pct and float(pct.group(1)) > 80:
        print('⚠️  rechazo >80%: el mapa esta practicamente CONGELADO. '
              'No leas la estabilidad, no significa nada.')


## 5c. TEST DE ACEPTACIÓN — calibración y la métrica del síntoma

`check_homography.py` mide dos cosas **separadas**, y hay que leerlas en este orden:

1. **ESTABILIDAD** — que el mapa no salte entre frames.
2. **CALIBRACIÓN** — que apunte a la cancha correcta. Un mapa *congelado* saca nota perfecta
   en estabilidad mientras pone al arquero a 39 m de su arco, así que estabilidad sola no
   dice nada.
3. **PELOTA DENTRO DEL ÁREA** — la métrica del síntoma que reportó Felipe. **Sólo se lee si
   la calibración dio "plausible"**: sobre la corrida del mapa congelado da 6,1%, que parece
   buenísimo y es un artefacto.

**Baseline a batir** (`..._crop (3).csv`, 18-ago 09:44):

| | hoy | objetivo |
|---|---|---|
| `x: p01` (con el arquero en cámara) | 4,6 m | ~0 |
| `x: p99` | 102,4 m | ~120 |
| detecciones fuera de los límites | 11% | <5% |
| saltos >1 m | 7,04% | menos |
| **pelota dentro del área** | **58,6%** | **~19,7%** |


In [ ]:
!python data_cleanup/check_homography.py --tracking-csv "{TRACKING_CSV}" --every 5

print('''
COMO LEERLO
  CALIBRACION "plausible"  + pelota en area cerca de 19,7%  -> cerrado, seguí
  CALIBRACION "MAL CALIBRADA"                               -> pegale esta salida
                                                               entera a Claude
''')

## 6. Generar eventos


In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'data_cleanup'))
from lib.match import Match
match = Match()
match.import_raw_data(os.path.dirname(TRACKING_CSV) + os.sep, os.path.basename(TRACKING_CSV))
print(f'Importados {match.frames} frames y {len(match.players)} objetos.')
events = match.generate_events()
print('Resumen:', events.summary())
EVENTS_CSV = os.path.join(EVENTS_DIR, VIDEO_NAME + '_events.csv')
events.export(path=EVENTS_DIR + os.sep, file_name=os.path.basename(EVENTS_CSV))
print('✅ Eventos:', EVENTS_CSV)


## 7. Chequeo — fragmentación de jugadores


In [ ]:
import csv
from collections import Counter
life = Counter()
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] == 'player': life[r['Object ID']] += 1
print('IDs de jugador distintos:', len(life), ' (menos = mejor; con el modelo chico eran ~184)')


## 8. Visualización


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Circle
df = pd.read_csv(EVENTS_CSV)
print(df['Type'].value_counts())

def draw_pitch(ax):
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,color='black',lw=2))
    ax.plot([0.5,0.5],[0,1],color='black',lw=1)
    ax.add_patch(Circle((0.5,0.5),0.083,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0.84,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
    ax.set_aspect(68.0/105.0); ax.axis('off')

pdf = df.dropna(subset=['Start X','Start Y'])
types = sorted(pdf['Type'].unique())
pal = list(plt.cm.tab10.colors)
colors = {t: pal[i % len(pal)] for i,t in enumerate(types)}
fig, ax = plt.subplots(figsize=(12,8))
ax.add_patch(plt.Rectangle((0,0),1,1,color='#3a8a3a',alpha=0.12,zorder=0)); draw_pitch(ax)
for t in types:
    s = pdf[pdf['Type']==t]
    ax.scatter(s['Start X'], s['Start Y'], s=120, color=colors[t], edgecolors='black', linewidths=0.6, label=f'{t} ({len(s)})', zorder=3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.02), ncol=4, frameon=False)
ax.set_title(f'Eventos detectados — {VIDEO_NAME}')
fig_path = os.path.join(EVENTS_DIR, VIDEO_NAME + '_event_map.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.show()
print('Mapa:', fig_path)


## 9. Descargar los CSV


In [ ]:
from google.colab import files

# Los TRES archivos son necesarios aguas abajo:
#   tracking.csv            -> el tracking
#   tracking.meta.json      -> fps efectivo (sin esto los tiempos salen mal)
#   *_ball_candidates.csv   -> candidatos de pelota para ball_viterbi.py
for path in (TRACKING_CSV,
             TRACKING_CSV.rsplit('.', 1)[0] + '.meta.json',
             TRACKING_CSV.rsplit('.', 1)[0] + '_ball_candidates.csv',
             EVENTS_CSV):
    if os.path.exists(path):
        files.download(path)
    else:
        print('FALTA:', path)

print('''
⚠️ Los archivos que hoy estan en ~/football_data/matches/clip-test/ son de la
   corrida con el MAPA CONGELADO (17-ago) y estan VIEJOS. Pisalos, no los
   mezcles: si quedan los dos, la cadena corre sobre datos viejos y da
   resultados byte-identicos sin avisar.

Guardalos asi (mismo nombre base, en la carpeta del partido):
  ~/football_data/matches/clip-test/tracking.csv
  ~/football_data/matches/clip-test/tracking.meta.json
  ~/football_data/matches/clip-test/tracking_ball_candidates.csv

Y borra los derivados viejos, que salen del CSV anterior:
  rm ~/football_data/matches/clip-test/tracking_{vit,clean,interp,clean_interp}.*

Despues, en la Mac:
  python3 data_cleanup/check_homography.py --tracking-csv ~/football_data/matches/clip-test/tracking.csv
''')